# 06_Decision_Points_Simulator

# 1 - Setup

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

BASE_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

DIR_03 = BASE_DIR / "03_Outputs"
DIR_04 = BASE_DIR / "04_Outputs"
DIR_05 = BASE_DIR / "05_Outputs"
DIR_06 = BASE_DIR / "06_Outputs"

DIR_06.mkdir(parents=True, exist_ok=True)

FILE_ACTIVITY_TRACKING = (
     DIR_03 / "Activity_Tracking.xlsx"
)

FILE_TRACKING = (
    DIR_04 / "Project_Tracking_with_Alarms.xlsx"
)

FILE_MASTER_MATERIALS = (
    DIR_05 / "Activity_Master_with_Materials.xlsx"
)

FILE_MATERIAL_ALARMS = (
    DIR_05 / "Material_Alarms.xlsx"
)

# ============================================================
# 2. COMPROBACIÓN DE ARCHIVOS
# ============================================================

files_required = {
    "Project Tracking": FILE_TRACKING,
    "Activity Master + Materials": FILE_MASTER_MATERIALS,
    "Material Alarms": FILE_MATERIAL_ALARMS,
    "Activity Tracking": FILE_ACTIVITY_TRACKING,
}

for name, path in files_required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"No se encuentra el archivo '{name}': {path}"
        )

    print(f"[OK] {name}: {path.name}")

# ============================================================
# 3. CARGA DE DATOS
# ============================================================

df_tracking = pd.read_excel(FILE_TRACKING)

df_master_materials = pd.read_excel(
    FILE_MASTER_MATERIALS
)

df_material_alarms = pd.read_excel(
    FILE_MATERIAL_ALARMS
)

df_activity_tracking = pd.read_excel(
    FILE_ACTIVITY_TRACKING
)

print("\n" + "=" * 65)
print("NOTEBOOK 06 — AUDITORÍA INICIAL")
print("=" * 65)

print(
    f"Periodos de seguimiento: {len(df_tracking):,}"
)

print(
    f"Actividades en Master + Materials: "
    f"{len(df_master_materials):,}"
)

print(
    f"Registros de alarmas de materiales: "
    f"{len(df_material_alarms):,}"
)

print("\nColumnas de seguimiento:")
print(df_tracking.columns.tolist())

print("\nColumnas del Activity Master:")
print(df_master_materials.columns.tolist())

print("\nColumnas de alarmas de materiales:")
print(df_material_alarms.columns.tolist())

print("\nColumnas del Activity Tracking:")
print(df_activity_tracking.columns.tolist())

# 2 - Auditoria
## 2.1 - Claves y seguridad

In [2]:
# ============================================================
# 4. AUDITORÍA DE CLAVES Y GRANULARIDAD
# ============================================================

required_master_cols = {
    "ID",
    "Name",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Is_Critical_Material",
    "Material_Type",
    "Material_Readiness_Date",
    "Supply_Delay_Days",
    "Material_Status",
    "Supply_Risk_Score",
    "Material_Alarm",
}

required_tracking_cols = {
    "TP",
    "Status_Date",
    "PV",
    "EV",
    "AC",
    "SPI",
    "CPI",
}

missing_master = (
    required_master_cols
    - set(df_master_materials.columns)
)

missing_tracking = (
    required_tracking_cols
    - set(df_tracking.columns)
)

if missing_master:
    raise ValueError(
        f"Faltan columnas en Activity Master: "
        f"{sorted(missing_master)}"
    )

if missing_tracking:
    raise ValueError(
        f"Faltan columnas en Project Tracking: "
        f"{sorted(missing_tracking)}"
    )

# Claves únicas
print(
    "IDs únicos en Activity Master:",
    df_master_materials["ID"].nunique()
)

print(
    "Filas duplicadas por ID:",
    df_master_materials["ID"].duplicated().sum()
)

print(
    "Periodos de seguimiento únicos:",
    df_tracking["TP"].nunique()
)

print(
    "Fechas de seguimiento válidas:",
    pd.to_datetime(
        df_tracking["Status_Date"],
        errors="coerce"
    ).notna().sum()
)

# Fechas
df_master_materials["Baseline_Start"] = pd.to_datetime(
    df_master_materials["Baseline_Start"],
    errors="coerce"
)

df_master_materials["Baseline_End"] = pd.to_datetime(
    df_master_materials["Baseline_End"],
    errors="coerce"
)

df_master_materials["Material_Readiness_Date"] = (
    pd.to_datetime(
        df_master_materials["Material_Readiness_Date"],
        errors="coerce"
    )
)

df_tracking["Status_Date"] = pd.to_datetime(
    df_tracking["Status_Date"],
    errors="coerce"
)

# Identificación de fechas incoherentes
invalid_baseline_dates = (
    df_master_materials["Baseline_Start"].notna()
    & df_master_materials["Baseline_End"].notna()
    & (
        df_master_materials["Baseline_End"]
        < df_master_materials["Baseline_Start"]
    )
)

print(
    "\nActividades con fin anterior al inicio:",
    invalid_baseline_dates.sum()
)

print(
    "Actividades sin fecha de inicio baseline:",
    df_master_materials["Baseline_Start"].isna().sum()
)

print(
    "Actividades sin fecha de fin baseline:",
    df_master_materials["Baseline_End"].isna().sum()
)

## 2.2 - Cobertura de Materiales

In [3]:
# ============================================================
# 5. AUDITORÍA DE COBERTURA DE MATERIALES
# ============================================================

if df_master_materials["ID"].duplicated().any():
    raise ValueError(
        "Activity Master contiene IDs duplicados. "
        "No se puede realizar el cruce hasta resolverlos."
    )

if df_material_alarms["ID"].duplicated().any():
    raise ValueError(
        "Material_Alarms contiene varios registros por ID. "
        "Revisar su granularidad antes de cruzar."
    )

material_cols = [
    "ID",
    "Is_Critical_Material",
    "Material_Type",
    "Material_Readiness_Date",
    "Supply_Delay_Days",
    "Material_Status",
    "Supply_Risk_Score",
    "Material_Alarm",
]

df_material_audit = df_master_materials[
    material_cols
].copy()

df_material_audit["ID"] = (
    df_material_audit["ID"].astype(str)
)

df_material_alarms["ID"] = (
    df_material_alarms["ID"].astype(str)
)

df_alarm_ids = set(df_material_alarms["ID"])

df_material_audit["Has_Material_Alarm"] = (
    df_material_audit["ID"].isin(df_alarm_ids)
)

print("\nCobertura de información de materiales")

print(
    "Actividades con material crítico:",
    df_material_audit[
        "Is_Critical_Material"
    ].eq(True).sum()
)

print(
    "Actividades con fecha de disponibilidad:",
    df_material_audit[
        "Material_Readiness_Date"
    ].notna().sum()
)

print(
    "Actividades con retraso de suministro informado:",
    df_material_audit[
        "Supply_Delay_Days"
    ].notna().sum()
)

print(
    "Actividades con alarma de material:",
    df_material_audit[
        "Has_Material_Alarm"
    ].sum()
)

print("\nDistribución de alarmas:")
print(
    df_material_audit[
        "Material_Alarm"
    ].value_counts(dropna=False)
)

print("\nEstado de materiales:")
print(
    df_material_audit[
        "Material_Status"
    ].value_counts(dropna=False)
)

## 2.3 - Granularidad Temporal

In [4]:
from pathlib import Path
import pandas as pd

# Usa la ruta que ya has definido en la configuración del Notebook 06.
# Si BASE_DIR ya existe, no hace falta volver a definirla.

print("=" * 75)
print("NOTEBOOK 06 — AUDITORÍA DE GRANULARIDAD TEMPORAL")
print("=" * 75)

extensions = {".csv", ".xlsx", ".xls", ".parquet"}

for file_path in sorted(BASE_DIR.rglob("*")):
    if not file_path.is_file():
        continue

    if file_path.suffix.lower() not in extensions:
        continue

    # Evitamos carpetas habituales de entornos y cachés.
    if any(part in {".git", ".venv", "venv", "__pycache__"}
           for part in file_path.parts):
        continue

    try:
        if file_path.suffix.lower() == ".csv":
            df = pd.read_csv(file_path, nrows=5)
            print(f"\nARCHIVO: {file_path.relative_to(BASE_DIR)}")
            print("Formato: CSV")
            print("Columnas:", df.columns.tolist())

        elif file_path.suffix.lower() in {".xlsx", ".xls"}:
            excel = pd.ExcelFile(file_path)

            for sheet in excel.sheet_names:
                df = pd.read_excel(
                    file_path,
                    sheet_name=sheet,
                    nrows=5
                )

                print(f"\nARCHIVO: {file_path.relative_to(BASE_DIR)}")
                print(f"Hoja: {sheet}")
                print("Columnas:", df.columns.tolist())

        elif file_path.suffix.lower() == ".parquet":
            df = pd.read_parquet(file_path)
            print(f"\nARCHIVO: {file_path.relative_to(BASE_DIR)}")
            print("Columnas:", df.columns.tolist())

    except Exception as exc:
        print(
            f"\nNo se pudo inspeccionar "
            f"{file_path.relative_to(BASE_DIR)}: {exc}"
        )

| Campo | Utilidad para el Notebook 06 |
|---|---|
| `TP` | Identificar el periodo de seguimiento |
| `Status_Date` | Establecer la fecha de corte |
| `Activity_ID` | Relacionar el registro con el Activity Master |
| `Actual_Start` | Examinar cuándo comenzó la actividad |
| `Actual_Duration` | Analizar su duración registrada |
| `Actual_Cost` | Examinar el coste acumulado registrado |
| `Pct_Complete` | Analizar su grado de ejecución |
| `EV` | Valor ganado de la actividad |
| `PV` | Valor planificado de la actividad |
| `Actual_Duration_Days` | Analizar la duración expresada en días |

Esta tabla es fundamental porque permite investigar la ejecución de cada actividad en distintos periodos, en lugar de depender exclusivamente de los indicadores agregados del proyecto.

No obstante, todavía no sabemos si todas las actividades tienen registros en los 120 periodos, si los costes y duraciones son acumulados o incrementales, ni cómo se representan las actividades que aún no han comenzado o ya han terminado.

Tampoco debemos confundir una actividad con un mal desempeño con una actividad que haya causado el retraso del proyecto. Para demostrar esto último necesitaremos considerar sus dependencias y, cuando sea posible, su relación con la ruta crítica.

## 2.4 - Calidad de Métricas

In [5]:
print("Dimensiones:", df_activity_tracking.shape)

print("\nColumnas:")
print(df_activity_tracking.columns.tolist())

display(df_activity_tracking.head(5))

In [6]:
print("Dimensiones:", df_activity_tracking.shape)

print("\nColumnas:")
print(df_activity_tracking.columns.tolist())

print("\nTipos de datos:")
display(df_activity_tracking.dtypes.to_frame("dtype"))

print("\nMuestra:")
display(df_activity_tracking.head())

### Auditoria de Actividades con duracion cero

Esta sección analiza los elementos con duración planificada cero para distinguir posibles hitos de actividades ejecutables y nodos estructurales. Se contrasta la duración con el coste planificado, la jerarquía WBS, las dependencias y el comportamiento registrado en el seguimiento.

El objetivo es evitar clasificaciones incorrectas antes de construir los indicadores y escenarios de decisión. Los elementos originales se conservan; su tratamiento depende del análisis posterior.

In [7]:
# Auditoría 04.1 — Distribución de duración

print("=== TIPOS DE DATOS ===")
display(df_activity_tracking.dtypes.to_frame("dtype"))

print("\n=== DISTRIBUCIÓN DE ACTUAL_DURATION_DAYS ===")
display(
    df_activity_tracking["Actual_Duration_Days"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("Actual_Duration_Days")
)

print("\n=== FRECUENCIA DE DURACIONES ===")
display(
    df_activity_tracking["Actual_Duration_Days"]
    .value_counts(dropna=False)
    .sort_index()
    .head(25)
    .to_frame("Registros")
)

In [8]:
# Auditoría 04.2 — Duración cero frente a estado de ejecución

df = df_activity_tracking.copy()

print("=== REGISTROS CON DURACIÓN CERO ===")
print(
    "Total:",
    (df["Actual_Duration_Days"] == 0).sum()
)

print("\n=== DURACIÓN CERO SEGÚN AVANCE ===")
df["Duracion_Cero"] = df["Actual_Duration_Days"].eq(0)

df["Estado_Avance"] = np.select(
    [
        df["Pct_Complete"].eq(0),
        df["Pct_Complete"].gt(0) & df["Pct_Complete"].lt(1),
        df["Pct_Complete"].ge(1)
    ],
    [
        "Sin avance",
        "En ejecución",
        "Completada"
    ],
    default="Sin clasificar"
)

display(
    pd.crosstab(
        df["Duracion_Cero"],
        df["Estado_Avance"],
        margins=True
    )
)

print("\n=== DURACIÓN CERO CON FECHA DE INICIO REGISTRADA ===")
df_cero_inicio = df[
    (df["Actual_Duration_Days"] == 0) &
    (df["Actual_Start"].notna())
]

print("Registros:", len(df_cero_inicio))
display(df_cero_inicio.head(15))

print("\n=== ACTIVIDADES COMPLETADAS CON DURACIÓN CERO ===")
df_cero_completadas = df[
    (df["Actual_Duration_Days"] == 0) &
    (df["Pct_Complete"] >= 1)
]

print("Registros:", len(df_cero_completadas))
display(df_cero_completadas.head(15))

**Situación	Registros**

|Duración cero y sin avance	|15.128

|Duración cero y actividad completada	|291

|Duración cero y actividad en ejecución	|0

|Duración positiva y actividad completada	|3.706

|Duración positiva y actividad en ejecución	|904

|Duración positiva y sin avance	|11

In [9]:
# Auditoría 04.3 — Identificación de actividades afectadas

df_cero_completadas = df_activity_tracking[
    (df_activity_tracking["Actual_Duration_Days"] == 0) &
    (df_activity_tracking["Pct_Complete"] >= 1)
].copy()

print("=== ACTIVIDADES AFECTADAS ===")
print("Registros:", len(df_cero_completadas))
print("IDs distintos:", df_cero_completadas["Activity_ID"].nunique())

display(
    df_cero_completadas.groupby("Activity_ID").agg(
        Registros=("TP", "count"),
        TP_Inicial=("TP", "min"),
        TP_Final=("TP", "max"),
        Fecha_Inicio=("Actual_Start", "min"),
        Fecha_Ultimo_Corte=("Status_Date", "max"),
        Duraciones_Originales=("Actual_Duration", lambda x: sorted(x.astype(str).unique())),
        Coste_Min=("Actual_Cost", "min"),
        Coste_Max=("Actual_Cost", "max"),
        EV_Min=("EV", "min"),
        EV_Max=("EV", "max")
    ).reset_index()
)

print("\n=== TRAYECTORIA DE LAS PRIMERAS ACTIVIDADES AFECTADAS ===")

ids_afectados = sorted(df_cero_completadas["Activity_ID"].unique())[:5]

display(
    df_activity_tracking[
        df_activity_tracking["Activity_ID"].isin(ids_afectados)
    ][
        [
            "TP", "Status_Date", "Activity_ID",
            "Actual_Start", "Actual_Duration",
            "Actual_Duration_Days", "Pct_Complete",
            "Actual_Cost", "EV", "PV"
        ]
    ].sort_values(["Activity_ID", "TP"])
)

In [10]:
# Auditoría 04.4 — Contraste con Activity Master

ids_afectados = sorted(
    df_cero_completadas["Activity_ID"].unique()
)

cols_master = [
    "ID",
    "Name",
    "WBS",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost"
]

display(
    df_master_materials[
        df_master_materials["ID"].isin(ids_afectados)
    ][cols_master]
    .sort_values("ID")
    .reset_index(drop=True)
)

In [11]:
# Auditoría 04.5 — Naturaleza de las actividades con duración cero

ids_cero_plan = (
    df_master_materials.loc[
        df_master_materials["Duration_Plan_Days"].eq(0),
        "ID"
    ]
)

cols = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan"
]

df_cero_plan = (
    df_master_materials[
        df_master_materials["ID"].isin(ids_cero_plan)
    ][cols]
    .sort_values("ID")
    .reset_index(drop=True)
)

print("Actividades con duración planificada cero:", len(df_cero_plan))
display(df_cero_plan)

In [12]:
# Auditoría 04.6 — Identificación de nodos de agregación WBS

df_master_check = df_master_materials.copy()

# Actividades con duración planificada cero
df_zero = df_master_check[
    df_master_check["Duration_Plan_Days"].eq(0)
].copy()

# Identificar descendientes mediante prefijo WBS
descendant_counts = []

for _, row in df_zero.iterrows():
    wbs = str(row["WBS"])

    descendants = df_master_check[
        df_master_check["WBS"].astype(str).str.startswith(wbs + ".")
    ]

    descendant_counts.append({
        "ID": row["ID"],
        "Name": row["Name"],
        "WBS": wbs,
        "WBS_Level": row["WBS_Level"],
        "N_Descendants": len(descendants),
        "Descendant_IDs": descendants["ID"].tolist()
    })

df_zero_hierarchy = pd.DataFrame(descendant_counts)

display(
    df_zero_hierarchy.sort_values(
        ["N_Descendants", "WBS"],
        ascending=[False, True]
    ).reset_index(drop=True)
)

print("\n=== RESUMEN ===")
print("Total filas duración cero:", len(df_zero_hierarchy))
print("Con descendientes:", (df_zero_hierarchy["N_Descendants"] > 0).sum())
print("Sin descendientes:", (df_zero_hierarchy["N_Descendants"] == 0).sum())

Ninguna de las 28 filas con duración planificada cero es un nodo de agregación WBS. Las 28 tienen cero descendientes. Por tanto, no podemos tratarlas como cabeceras de grupos ni propagar sus nombres a las actividades subordinadas.

Esto significa que tenemos dos conceptos distintos:

- Agregación jerárquica: se identifica mediante la estructura WBS y sus descendientes, independientemente de que tenga duración o coste.
- 
- Hito o elemento de duración cero: es una fila terminal que puede tener una función en la planificación y en las dependencias, aunque no consuma duración ni presupuesto.

In [13]:
# Auditoría 04.7 — Comportamiento de los elementos de duración cero

zero_ids = df_zero_hierarchy["ID"].tolist()

df_zero_tracking = df_activity_tracking[
    df_activity_tracking["Activity_ID"].isin(zero_ids)
].copy()

df_zero_summary = (
    df_zero_tracking
    .groupby("Activity_ID")
    .agg(
        Name=("Activity_ID", "first"),
        Registros=("TP", "count"),
        Con_inicio=("Actual_Start", "count"),
        TPs_con_avance=("Pct_Complete", lambda x: (x > 0).sum()),
        TPs_completados=("Pct_Complete", lambda x: (x >= 1).sum()),
        Duracion_max=("Actual_Duration_Days", "max"),
        Coste_max=("Actual_Cost", "max"),
        EV_max=("EV", "max")
    )
    .reset_index()
)

df_zero_summary = df_zero_summary.merge(
    df_master_materials[["ID", "Name", "Predecessors", "Successors"]],
    left_on="Activity_ID",
    right_on="ID",
    how="left"
)

display(
    df_zero_summary[
        [
            "Activity_ID", "Name_y", "Registros",
            "Con_inicio", "TPs_con_avance", "TPs_completados",
            "Duracion_max", "Coste_max", "EV_max",
            "Predecessors", "Successors"
        ]
    ].rename(columns={"Name_y": "Name"})
    .sort_values("Activity_ID")
    .reset_index(drop=True)
)

In [14]:
# Auditoría 04.8 — Contexto jerárquico de elementos con duración cero

zero_ids = set(df_zero_hierarchy["ID"])

df_context = df_master_materials.copy()
df_context["WBS"] = df_context["WBS"].astype(str)

rows = []

for _, row in df_context[df_context["ID"].isin(zero_ids)].iterrows():

    wbs = row["WBS"]
    parent_wbs = ".".join(wbs.split(".")[:-1])

    siblings = df_context[
        df_context["WBS"].apply(
            lambda x: x.rsplit(".", 1)[0] == parent_wbs
        )
    ]

    rows.append({
        "ID": row["ID"],
        "Name": row["Name"],
        "WBS": wbs,
        "Parent_WBS": parent_wbs,
        "Sibling_IDs": siblings["ID"].tolist(),
        "Sibling_Names": siblings["Name"].tolist(),
        "Sibling_Durations": siblings["Duration_Plan_Days"].tolist()
    })

df_zero_context = pd.DataFrame(rows)

display(
    df_zero_context.sort_values("WBS").reset_index(drop=True)
)

| IDs | WBS padre | Elementos hermanos | Interpretación |
| :--- | :--- | :--- | :--- |
| 3, 4, 5 | 1.1.1 | Milieuvergunning, Bouwvergunning, RUP | Tres elementos de permisos |
| 52–88 | 1.1.5.1.1 a 1.1.5.1.7 | Pre installation + instalación y pruebas | Hito inicial de cada secuencia de montaje |
| 110–116 | 1.1.5.4 | WEC 1–7 | Siete elementos asociados a aerogeneradores |
| 142–148 | 1.1.5.8 | WEC 1–7 | Otra secuencia de siete elementos WEC |
| 152, 159 | 1.1.6.1 | Actividades de construcción eléctrica | Elementos dentro de una secuencia de trabajo |
| 162 | 1.1.7.1 | Order connection, Order material, Lay cable, etc. | Elemento inicial de la secuencia de conexión |
| 166 | 1.2 | Wind Farm, Wind Farm, Finish | Hito final del proyecto |

Hay un matiz especialmente interesante: los elementos Pre installation tienen duración cero, pero sus hermanos tienen duraciones de 10, 2, 2 y 2 días. Es decir, no son cabeceras WBS, sino registros que forman parte de una secuencia operativa.
2. Cómo lo modelaría
No trasladaría automáticamente los nombres de estos registros a una columna informativa común. Eso podría generar interpretaciones erróneas.

En su lugar, construiría un modelo con dos niveles de información:

A. Maestro estructural

Conservaría todos los registros originales, sus identificadores, WBS, fechas y dependencias.

B. Maestro operativo normalizado

Añadiría una clasificación que permita distinguir:
- ACTIVITY: actividad con duración o coste planificado.
- MILESTONE: hito o evento de duración cero que participa en la red.
- INFORMATIONAL: elemento sin ejecución ni función operativa acreditada.
- SUMMARY: nodo jerárquico con descendientes.
En nuestro caso, los nodos SUMMARY se identificarán mediante descendientes WBS, no mediante duración cero.

Y mantendría las dependencias en una tabla independiente, de forma que podamos excluir un elemento de determinados cálculos sin romper la red de planificación.
3. Antes de transformar: una última comprobación
Hay una cuestión que quiero resolver antes de asignar las categorías: ¿los elementos de duración cero que aparecen como primeros hermanos representan hitos reales del proyecto o son registros administrativos utilizados para organizar la planificación?

Podemos obtener una primera evidencia comparando sus fechas con las de las actividades hermanas y sus dependencias.

In [15]:
# Auditoría 04.9 — Secuencias de preinstalación

ids_preinstallation = [52, 58, 64, 70, 76, 82, 88]

ids_sequence = []

for activity_id in ids_preinstallation:
    row = df_master_materials.loc[
        df_master_materials["ID"].eq(activity_id)
    ].iloc[0]

    parent_wbs = str(row["WBS"]).rsplit(".", 1)[0]

    siblings = df_master_materials[
        df_master_materials["WBS"].astype(str).apply(
            lambda x: x.rsplit(".", 1)[0] == parent_wbs
        )
    ]

    ids_sequence.extend(siblings["ID"].tolist())

df_preinstallation_sequence = (
    df_master_materials[
        df_master_materials["ID"].isin(set(ids_sequence))
    ][
        [
            "ID", "Name", "WBS",
            "Baseline_Start", "Baseline_End",
            "Duration_Plan_Days",
            "Predecessors", "Successors"
        ]
    ]
    .sort_values(["WBS", "Baseline_Start"])
    .reset_index(drop=True)
)

display(df_preinstallation_sequence)

Funcionan como hitos de inicio de cada secuencia de montaje, y además están integrados explícitamente en la red de dependencias.

Fíjate en la secuencia de la primera torre:
| Tipo | Ejemplo | Tratamiento |
| :--- | :--- | :--- |
| Actividad | Installation tower | Cálculos de duración, coste y avance |
| Hito de inicio | Pre installation | Seguimiento de fecha y dependencias |
| Hito de finalización | Finish (ID56, 62, 68...) | Seguimiento de fecha y dependencias |
| Elemento sin ejecución registrada | Milieuvergunning, Bouwvergunning | Pendiente de determinar su función |
| Nodo de agregación | WBS con descendientes | Consolidación jerárquica |

La estructura se repite en las siete secuencias:
- El hito Pre installation tiene duración cero.
- La actividad Installation tower comienza al siguiente día laborable, a las 08:00.
- La relación FS conecta ambos elementos.
- El hito también puede tener relaciones SF con otras actividades del proyecto.
Esto es consistente con un hito de planificación, no con una cabecera WBS.
1. La distinción que debemos aplicar
Tenemos ya evidencia para distinguir:
Tipo	Ejemplo	Tratamiento
Actividad	Installation tower	Cálculos de duración, coste y avance
Hito de inicio	Pre installation	Seguimiento de fecha y dependencias
Hito de finalización	Finish (ID56, 62, 68...)	Seguimiento de fecha y dependencias
Elemento sin ejecución registrada	Milieuvergunning, Bouwvergunning	Pendiente de determinar su función
Nodo de agregación	WBS con descendientes	Consolidación jerárquica

Una precisión: el nombre Finish no basta para clasificar todos los registros con ese nombre como hitos. Tenemos que atender a su duración, posición y relaciones.
2. Qué implica para la limpieza de datos
Mi recomendación es que no eliminemos los registros de duración cero de Activity_Master ni de Activity_Tracking.

En cambio, podemos separar los cálculos:
- Duración planificada: agregar actividades operativas; no sumar hitos.
- Coste planificado: agregar los elementos con presupuesto, evitando duplicar niveles jerárquicos.
- Avance: conservar el seguimiento de hitos, pero no mezclarlos automáticamente con el avance ponderado por coste.
- Dependencias: mantener todos los nodos y relaciones, incluidos los hitos.
Así evitamos que los ceros distorsionen estadísticas de duración, sin perder información relevante para la planificación.
3. Siguiente paso: comprobar los hitos Finish
Antes de construir el clasificador, quiero verificar si los registros Finish de estas secuencias presentan el mismo patrón que Pre installation.

In [16]:
# Auditoría 04.10 — Hitos Finish de las secuencias de montaje

df_finish = df_master_materials[
    df_master_materials["Name"].str.strip().str.lower().eq("finish")
].copy()

display(
    df_finish[
        [
            "ID", "Name", "WBS",
            "Baseline_Start", "Baseline_End",
            "Duration_Plan_Days",
            "Total_Cost_Plan",
            "Predecessors", "Successors"
        ]
    ].sort_values("WBS").reset_index(drop=True)
)

In [17]:
# 04.11 — Inspección de permisos y actividades sucesoras
# Solo lectura: no modifica datos ni archivos.

ids_to_inspect = [3, 4, 5, 16, 17]

cols_to_show = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
]

display(
    df_master_materials.loc[
        df_master_materials["ID"].isin(ids_to_inspect),
        cols_to_show
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

print("\n=== SEGUIMIENTO DE LOS CINCO ELEMENTOS ===")

display(
    df_activity_tracking.loc[
        df_activity_tracking["Activity_ID"].isin(ids_to_inspect)
    ]
    .groupby("Activity_ID")
    .agg(
        Registros=("TP", "size"),
        Con_inicio=("Actual_Start", "count"),
        TPs_con_avance=("Pct_Complete", lambda s: (s.fillna(0) > 0).sum()),
        TPs_completados=("Pct_Complete", lambda s: (s.fillna(0) >= 1).sum()),
        Duracion_max=("Actual_Duration_Days", "max"),
        Coste_max=("Actual_Cost", "max"),
        EV_max=("EV", "max"),
    )
    .reset_index()
)

| ID | Actividad | Duración planificada | Coste planificado | Seguimiento | Interpretación provisional |
|---:|---|---:|---:|---|---|
| 3 | Milieuvergunning | 0 | 0 € | Sin inicio ni avance | Posible hito o registro administrativo |
| 4 | Bouwvergunning | 0 | 0 € | Sin inicio ni avance | Posible hito o registro administrativo |
| 5 | RUP | 0 | 0 € | Sin inicio ni avance | Elemento de duración cero que precede a otras actividades |
| 16 | Roads and hardstands | 14 días | 16.511,74 € | Seguimiento registrado | Actividad con duración y coste |
| 17 | contracten | 14 días | 16.511,74 € | Seguimiento registrado | Actividad con duración y coste |

In [18]:
# 04.12 — Clasificación preliminar de elementos de duración cero
# Solo lectura: no modifica datos ni archivos.

zero_duration = df_master_materials.loc[
    df_master_materials["Duration_Plan_Days"].eq(0)
].copy()

print(f"Elementos con duración planificada cero: {len(zero_duration)}")

cols = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Total_Cost_Plan",
    "Material_Type",
    "Criticality",
    "Material_Status",
    "Material_Alarm",
]

display(
    zero_duration[
        [c for c in cols if c in zero_duration.columns]
    ]
    .sort_values(["WBS", "ID"])
    .reset_index(drop=True)
)

| Grupo | IDs | Elementos | Interpretación provisional |
|---|---|---|---|
| Permisos y planificación administrativa | 3, 4, 5 | Milieuvergunning, Bouwvergunning, RUP | Posibles hitos o registros administrativos. El ID 5 participa en la red de dependencias. |
| Preparación de la instalación de torres | 52, 58, 64, 70, 76, 82, 88 | Pre installation | Hitos de duración cero vinculados a actividades posteriores y, en algunos casos, a relaciones con las cimentaciones. |
| Posiciones WEC de la fase 1 | 110–116 | WEC 1–7 | Elementos de duración cero con predecesores y sucesores. Su función exacta debe confirmarse. |
| Posiciones WEC de la fase 2 | 142–148 | WEC 1–7 | Elementos de duración cero; el ID 148 precede al cierre del proyecto. |
| Otras actividades de duración cero | 152, 159, 162 | Excavation for cabin, Energize, Order connection | Posibles hitos o actividades modeladas como eventos; requieren revisar su significado funcional. |
| Cierre del proyecto | 166 | Finish | Elemento terminal de duración cero, conectado con el ID 148. |

In [19]:
# 04.13 — Contexto de los elementos WEC
# Solo lectura: no modifica datos ni archivos.

ids_wec = list(range(102, 125)) + list(range(134, 149))

cols_wec = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
]

display(
    df_master_materials.loc[
        df_master_materials["ID"].isin(ids_wec),
        cols_wec
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

Los elementos WEC de duración cero no son simples títulos ni agregaciones: forman parte de una secuencia de planificación por aerogenerador.
WBS	Fase	Actividades

1.1.5.3	Instalación	WEC 1–7 (IDs 102–108)

1.1.5.4	Energizing	WEC 1–7 (IDs 110–116)

1.1.5.5	Commissioning	WEC 1–7 (IDs 118–124)

1.1.5.7	Actividad anterior a Take over	WEC 1–7 (IDs 134–140)

1.1.5.8	Take over	WEC 1–7 (IDs 142–148)

    

In [20]:
# 04.14 — Contexto de hitos y actividades singulares
# Solo lectura.

ids_context = [151, 152, 153, 154, 158, 159, 160, 161, 162, 163, 164, 165, 166]

cols_context = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
]

display(
    df_master_materials.loc[
        df_master_materials["ID"].isin(ids_context),
        cols_context
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

In [21]:
# 04.15 — Secuencia completa de cimentación e instalación WEC 1
# Solo lectura.

ids_sequence = list(range(42, 57))

cols_sequence = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
]

display(
    df_master_materials.loc[
        df_master_materials["ID"].isin(ids_sequence),
        cols_sequence
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

In [22]:
# 04.16 — Verificación de datos originales del ID 56
# Solo lectura.

display(
    df_master_materials.loc[
        df_master_materials["ID"].eq(56)
    ].T
)

print("\n=== VALORES Y TIPOS ===")

for col in [
    "Duration_Plan",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost",
]:
    if col in df_master_materials.columns:
        print(
            f"{col}: "
            f"{df_master_materials.loc[df_master_materials['ID'].eq(56), col].iloc[0]!r}"
        )

### Validación de la jerarquía y los costes del proyecto

Se comprueba la coherencia entre los nodos resumen y sus descendientes, tanto inmediatos como terminales. Esta validación permite identificar la estructura de agregación y evitar la doble contabilización de costes al consolidar los resultados del proyecto.

In [23]:
# 04.17 — Auditoría global de estructura y atributos del maestro
# Solo lectura: no modifica datos ni archivos.

df_audit = df_master_materials.copy()

df_audit["Has_Duration"] = (
    df_audit["Duration_Plan_Days"].fillna(0) > 0
)

df_audit["Has_Cost"] = (
    df_audit["Total_Cost_Plan"].fillna(0) > 0
)

df_audit["Has_Predecessors"] = (
    df_audit["Predecessors"].notna()
    & df_audit["Predecessors"].astype(str).str.strip().ne("")
)

df_audit["Has_Successors"] = (
    df_audit["Successors"].notna()
    & df_audit["Successors"].astype(str).str.strip().ne("")
)

df_audit["Has_Dependencies"] = (
    df_audit["Has_Predecessors"]
    | df_audit["Has_Successors"]
)

print("=== DISTRIBUCIÓN POR NIVEL WBS ===")

display(
    df_audit.groupby("WBS_Level")
    .agg(
        Elements=("ID", "count"),
        With_Duration=("Has_Duration", "sum"),
        Without_Duration=("Has_Duration", lambda s: (~s).sum()),
        With_Cost=("Has_Cost", "sum"),
        Without_Cost=("Has_Cost", lambda s: (~s).sum()),
        With_Dependencies=("Has_Dependencies", "sum"),
    )
    .reset_index()
)

print("\n=== COMBINACIÓN DE DURACIÓN, COSTE Y DEPENDENCIAS ===")

display(
    df_audit.groupby(
        ["Has_Duration", "Has_Cost", "Has_Dependencies"],
        dropna=False
    )
    .agg(Elements=("ID", "count"))
    .reset_index()
    .sort_values("Elements", ascending=False)
)

print("\n=== TOTALES ===")

print(f"Registros: {len(df_audit)}")
print(f"Duración cero o nula: {(~df_audit['Has_Duration']).sum()}")
print(f"Coste cero o nulo: {(~df_audit['Has_Cost']).sum()}")
print(f"Con dependencias: {df_audit['Has_Dependencies'].sum()}")

In [24]:
# 04.18 — Registros con duración positiva y coste cero
# Solo lectura.

mask = (
    (df_audit["Duration_Plan_Days"].fillna(0) > 0)
    & (df_audit["Total_Cost_Plan"].fillna(0) == 0)
)

cols = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost",
]

display(
    df_audit.loc[mask, cols]
    .sort_values(["WBS_Level", "WBS", "ID"])
    .reset_index(drop=True)
)

In [25]:
# 04.19 — Comprobar descendientes WBS de los tres resúmenes
# Solo lectura; no modifica el maestro.

summary_ids = [2, 109, 141]

for summary_id in summary_ids:
    row = df_master_materials.loc[
        df_master_materials["ID"].eq(summary_id)
    ].iloc[0]

    wbs = str(row["WBS"]).strip()

    descendants = df_master_materials.loc[
        df_master_materials["WBS"]
        .astype(str)
        .str.strip()
        .str.startswith(wbs + ".")
    ].copy()

    print(
        f"\n{'=' * 75}\n"
        f"ID {summary_id} | {row['Name']} | WBS {wbs}\n"
        f"Descendientes encontrados: {len(descendants)}"
    )

    cols = [
        "ID",
        "Name",
        "WBS",
        "WBS_Level",
        "Duration_Plan_Days",
        "Total_Cost_Plan",
        "Predecessors",
        "Successors",
    ]

    display(
        descendants[cols]
        .sort_values(["WBS_Level", "WBS", "ID"])
        .reset_index(drop=True)
    )

In [26]:
# 04.20 — Identificación sistemática de elementos con descendientes WBS
# No modifica el maestro ni los archivos originales.

df_wbs_audit = df_master_materials.copy()

df_wbs_audit["WBS_norm"] = (
    df_wbs_audit["WBS"]
    .astype("string")
    .str.strip()
)

summary_candidates = []

for _, row in df_wbs_audit.iterrows():
    wbs = row["WBS_norm"]

    if pd.isna(wbs) or wbs == "":
        continue

    descendants = df_wbs_audit[
        df_wbs_audit["WBS_norm"].str.startswith(
            wbs + ".", na=False
        )
    ]

    if not descendants.empty:
        summary_candidates.append({
            "ID": row["ID"],
            "Name": row["Name"],
            "WBS": wbs,
            "WBS_Level": row["WBS_Level"],
            "N_Descendants": len(descendants),
            "Duration_Plan_Days": row["Duration_Plan_Days"],
            "Total_Cost_Plan": row["Total_Cost_Plan"],
        })

df_summary_candidates = (
    pd.DataFrame(summary_candidates)
    .sort_values(["WBS_Level", "WBS"])
    .reset_index(drop=True)
)

print(
    "Elementos con descendientes WBS:",
    len(df_summary_candidates)
)

display(df_summary_candidates)

In [27]:
# 04.21 — Consistencia económica de los candidatos a resumen WBS
# Solo lectura. No modifica los datos originales.

df_cost_audit = df_master_materials.copy()

df_cost_audit["WBS_norm"] = (
    df_cost_audit["WBS"]
    .astype("string")
    .str.strip()
)

df_cost_audit["Total_Cost_Plan"] = pd.to_numeric(
    df_cost_audit["Total_Cost_Plan"],
    errors="coerce"
)

cost_rows = []

for _, parent in df_cost_audit.iterrows():
    parent_wbs = parent["WBS_norm"]

    if pd.isna(parent_wbs) or parent_wbs == "":
        continue

    descendants = df_cost_audit[
        df_cost_audit["WBS_norm"].str.startswith(
            parent_wbs + ".", na=False
        )
    ].copy()

    if descendants.empty:
        continue

    immediate_prefix = parent_wbs + "."

    descendants["Relative_WBS"] = (
        descendants["WBS_norm"]
        .str[len(immediate_prefix):]
    )

    immediate_children = descendants[
        ~descendants["Relative_WBS"].str.contains(
            r"\.", regex=True, na=False
        )
    ]

    parent_cost = parent["Total_Cost_Plan"]

    immediate_cost = immediate_children[
        "Total_Cost_Plan"
    ].sum(min_count=1)

    descendant_cost = descendants[
        "Total_Cost_Plan"
    ].sum(min_count=1)

    cost_rows.append({
        "ID": parent["ID"],
        "Name": parent["Name"],
        "WBS": parent_wbs,
        "WBS_Level": parent["WBS_Level"],
        "N_Immediate_Children": len(immediate_children),
        "N_All_Descendants": len(descendants),
        "Parent_Cost": parent_cost,
        "Immediate_Children_Cost": immediate_cost,
        "All_Descendants_Cost": descendant_cost,
        "Diff_vs_Immediate": (
            parent_cost - immediate_cost
            if pd.notna(parent_cost) and pd.notna(immediate_cost)
            else np.nan
        ),
        "Diff_vs_All_Descendants": (
            parent_cost - descendant_cost
            if pd.notna(parent_cost) and pd.notna(descendant_cost)
            else np.nan
        ),
    })

df_summary_cost_audit = pd.DataFrame(cost_rows)

display(
    df_summary_cost_audit.sort_values(
        ["WBS_Level", "WBS"]
    ).reset_index(drop=True)
)

In [28]:
# 04.22 — Validación de costes de resúmenes frente a descendientes terminales
# Solo lectura; no modifica el maestro.

df_terminal_audit = df_master_materials.copy()

df_terminal_audit["WBS_norm"] = (
    df_terminal_audit["WBS"]
    .astype("string")
    .str.strip()
)

df_terminal_audit["Total_Cost_Plan"] = pd.to_numeric(
    df_terminal_audit["Total_Cost_Plan"],
    errors="coerce"
)

summary_ids = set(df_summary_candidates["ID"])

terminal_cost_rows = []

for _, parent in df_terminal_audit[
    df_terminal_audit["ID"].isin(summary_ids)
].iterrows():

    parent_wbs = parent["WBS_norm"]

    descendants = df_terminal_audit[
        df_terminal_audit["WBS_norm"].str.startswith(
            parent_wbs + ".", na=False
        )
    ].copy()

    descendant_wbs = descendants["WBS_norm"]

    descendants["Has_Children"] = descendant_wbs.apply(
        lambda wbs: (
            df_terminal_audit["WBS_norm"]
            .str.startswith(str(wbs) + ".", na=False)
            .any()
        )
    )

    terminal_descendants = descendants[
        ~descendants["Has_Children"]
    ]

    terminal_cost = terminal_descendants[
        "Total_Cost_Plan"
    ].sum(min_count=1)

    parent_cost = parent["Total_Cost_Plan"]

    terminal_cost_rows.append({
        "ID": parent["ID"],
        "Name": parent["Name"],
        "WBS": parent_wbs,
        "Parent_Cost": parent_cost,
        "Terminal_Descendants": len(terminal_descendants),
        "Terminal_Cost": terminal_cost,
        "Difference": (
            parent_cost - terminal_cost
            if pd.notna(parent_cost) and pd.notna(terminal_cost)
            else np.nan
        ),
    })

df_terminal_cost_audit = pd.DataFrame(terminal_cost_rows)

display(
    df_terminal_cost_audit.sort_values(
        ["WBS"]
    ).reset_index(drop=True)
)

### Reconciliación de EV, PV y AC

Se agregan por periodo los valores de las actividades terminales y se comparan con los indicadores del seguimiento global del proyecto. 

La reconciliación verifica que la base analítica reproduce los valores agregados originales antes de utilizarla para identificar desviaciones y evaluar escenarios.

In [29]:

# ============================================================
# CIERRE 1 — CLASIFICACIÓN OPERATIVA DEL ACTIVITY MASTER
# ============================================================

df_master_operativo = df_master_materials.copy()

# Normalizar WBS para identificar descendientes
df_master_operativo["WBS_norm"] = (
    df_master_operativo["WBS"].astype("string").str.strip()
)

# Identificar nodos que tienen descendientes
wbs_values = df_master_operativo["WBS_norm"].dropna().tolist()

has_children = []
for wbs in df_master_operativo["WBS_norm"]:
    if pd.isna(wbs):
        has_children.append(False)
    else:
        has_children.append(
            any(
                other.startswith(str(wbs) + ".")
                for other in wbs_values
            )
        )

df_master_operativo["WBS_Has_Children"] = has_children

# Clasificación estructural, sin atribuir naturaleza no demostrada
df_master_operativo["Element_Type"] = np.select(
    [
        df_master_operativo["WBS_Has_Children"],
        df_master_operativo["Duration_Plan_Days"].eq(0)
    ],
    [
        "SUMMARY",
        "ZERO_DURATION_REVIEW"
    ],
    default="TERMINAL_ACTIVITY"
)

df_master_operativo["Is_Terminal"] = (
    ~df_master_operativo["WBS_Has_Children"]
)

df_master_operativo["Is_Executable"] = (
    df_master_operativo["Element_Type"].eq("TERMINAL_ACTIVITY")
)

# Resumen de clasificación
print("=== CLASIFICACIÓN OPERATIVA ===")
display(
    df_master_operativo.groupby("Element_Type")
    .agg(
        Elements=("ID", "count"),
        Planned_Cost=("Total_Cost_Plan", "sum"),
        Planned_Duration=("Duration_Plan_Days", "sum")
    )
    .reset_index()
)

# Comprobaciones de cierre
print("\n=== VALIDACIONES ===")
print("Registros:", len(df_master_operativo))
print("IDs duplicados:", df_master_operativo["ID"].duplicated().sum())
print("Nodos resumen:", df_master_operativo["WBS_Has_Children"].sum())
print("Elementos terminales:", df_master_operativo["Is_Terminal"].sum())
print("Elementos de duración cero:", 
      df_master_operativo["Duration_Plan_Days"].eq(0).sum())

# Coste reconstruido desde elementos terminales
cost_terminal = df_master_operativo.loc[
    df_master_operativo["Is_Terminal"], "Total_Cost_Plan"
].sum()

cost_project = df_master_operativo.loc[
    df_master_operativo["ID"].eq(0), "Total_Cost_Plan"
].iloc[0]

print(f"\nCoste terminal: {cost_terminal:,.2f} €")
print(f"Coste proyecto (ID 0): {cost_project:,.2f} €")
print(f"Diferencia: {cost_terminal - cost_project:,.8f} €")

assert len(df_master_operativo) == 167
assert df_master_operativo["ID"].is_unique
assert df_master_operativo["WBS_Has_Children"].sum() == 32
assert df_master_operativo["Is_Terminal"].sum() == 135
assert df_master_operativo["Duration_Plan_Days"].eq(0).sum() == 28
assert np.isclose(cost_terminal, cost_project, atol=0.01)

print("\n[OK] Clasificación estructural y agregación de costes.")



In [30]:

# ============================================================
# CIERRE 2 — RECONCILIACIÓN EV / PV / AC
# Una única versión consolidada
# ============================================================

# 1. Actividades terminales
terminal_ids = set(
    df_master_operativo.loc[
        df_master_operativo["Is_Terminal"], "ID"
    ]
)

# 2. Tracking de actividades terminales
df_act = df_activity_tracking.copy()

df_act["Activity_ID"] = pd.to_numeric(
    df_act["Activity_ID"], errors="coerce"
)

df_act_terminal = df_act.loc[
    df_act["Activity_ID"].isin(terminal_ids)
].copy()

# 3. Agregación por periodo
df_agg = (
    df_act_terminal
    .groupby("TP", as_index=False)
    .agg(
        EV_terminal=("EV", "sum"),
        PV_terminal=("PV", "sum"),
        AC_terminal=("Actual_Cost", "sum")
    )
)

# 4. Seguimiento agregado del proyecto
df_project = (
    df_tracking[["TP", "EV", "PV", "AC"]]
    .rename(columns={
        "EV": "EV_project",
        "PV": "PV_project",
        "AC": "AC_project"
    })
)

# 5. Conciliación
df_check = df_project.merge(
    df_agg,
    on="TP",
    how="outer",
    validate="one_to_one",
    indicator=True
)

for metric in ["EV", "PV", "AC"]:
    df_check[f"Diff_{metric}"] = (
        df_check[f"{metric}_terminal"]
        - df_check[f"{metric}_project"]
    )

# 6. Resultado
print("=== RECONCILIACIÓN EV / PV / AC ===")
print("Periodos:", len(df_check))
print("Correspondencias:", df_check["_merge"].eq("both").sum())

for metric in ["EV", "PV", "AC"]:
    print(
        f"Diferencia máxima {metric}: "
        f"{df_check[f'Diff_{metric}'].abs().max():,.8f} €"
    )

# 7. Validaciones necesarias para dar por válida la base
assert len(df_check) == 120
assert df_check["_merge"].eq("both").all()

for metric in ["EV", "PV", "AC"]:
    assert np.allclose(
        df_check[f"Diff_{metric}"],
        0,
        atol=0.01
    ), f"Error en la conciliación de {metric}"

print("\n[OK] Las actividades terminales reproducen EV, PV y AC.")


In [31]:
"""
# 1. Preparar el seguimiento por actividad
df_act = df_activity_tracking.copy()

# Normalizar clave de actividad
df_act["Activity_ID"] = pd.to_numeric(
    df_act["Activity_ID"], errors="coerce"
)

# Filtrar las 135 actividades terminales
df_act_terminal = df_act[
    df_act["Activity_ID"].isin(terminal_ids)
].copy()

# 2. Agregar EV, PV y AC por periodo
df_agg = (
    df_act_terminal
    .groupby("TP", as_index=False)[["EV", "PV", "Actual_Cost"]]
    .sum(min_count=1)
    .rename(columns={"Actual_Cost": "AC_terminal"})
)

# 3. Comparar con el seguimiento agregado del proyecto
df_check = df_tracking[
    ["TP", "EV", "PV", "AC"]
].merge(
    df_agg,
    on="TP",
    how="outer",
    validate="one_to_one"
)

for metric in ["EV", "PV"]:
    df_check[f"Diff_{metric}"] = (
        df_check[f"{metric}_terminal"] - df_check[metric]
        if f"{metric}_terminal" in df_check.columns
        else np.nan
    )

df_check["Diff_AC"] = (
    df_check["AC_terminal"] - df_check["AC"]
)

# 4. Resultado
print("Periodos comparados:", len(df_check))
print("Periodos sin agregado terminal:", df_check["EV_terminal"].isna().sum()
      if "EV_terminal" in df_check.columns else "Revisar")

display(df_check.tail(10))
"""

In [32]:
"""
df_check = df_check.rename(columns={
    "EV_x": "EV_project",
    "EV_y": "EV_terminal",
    "PV_x": "PV_project",
    "PV_y": "PV_terminal"
})

df_check["Diff_EV"] = (
    df_check["EV_terminal"] - df_check["EV_project"]
)
df_check["Diff_PV"] = (
    df_check["PV_terminal"] - df_check["PV_project"]
)
df_check["Diff_AC"] = (
    df_check["AC_terminal"] - df_check["AC"]
)

print("=== VALIDACIÓN EV / PV / AC ===")
print("Periodos:", len(df_check))
print("Máxima diferencia EV:", df_check["Diff_EV"].abs().max())
print("Máxima diferencia PV:", df_check["Diff_PV"].abs().max())
print("Máxima diferencia AC:", df_check["Diff_AC"].abs().max())

assert np.allclose(df_check["Diff_EV"], 0, atol=0.01)
assert np.allclose(df_check["Diff_PV"], 0, atol=0.01)
assert np.allclose(df_check["Diff_AC"], 0, atol=0.01)

print("[OK] Agregación terminal consistente en los 120 periodos.")
"""

In [33]:
# ============================================================
# COMPROBACIÓN 1/3 — COHERENCIA TEMPORAL EV / PV / AC
# ============================================================

df_temp = df_tracking[
    ["TP", "Status_Date", "EV", "PV", "AC"]
].copy()

df_temp = df_temp.sort_values("TP").reset_index(drop=True)

# 1. Un registro por periodo
print("=== CLAVES Y ORDEN TEMPORAL ===")
print("Filas:", len(df_temp))
print("TP únicos:", df_temp["TP"].nunique())
print("TP duplicados:", df_temp["TP"].duplicated().sum())
print("Fechas no crecientes:",
      (df_temp["Status_Date"].diff().dropna() <= pd.Timedelta(0)).sum())

# 2. Valores nulos
metrics = ["EV", "PV", "AC"]

print("\n=== VALORES NULOS ===")
print(df_temp[metrics].isna().sum())

# 3. Evolución acumulada: detectar retrocesos
print("\n=== RETROCESOS ENTRE PERIODOS ===")

for col in metrics:
    delta = df_temp[col].diff()

    decreases = df_temp.loc[
        delta < -0.01,
        ["TP", "Status_Date", col]
    ].copy()

    print(f"\n{col}: {len(decreases)} retrocesos")

    if not decreases.empty:
        display(decreases.head(10))

# 4. Incrementos máximos y mínimos
print("\n=== VARIACIONES ENTRE PERIODOS ===")

for col in metrics:
    delta = df_temp[col].diff()

    print(
        f"{col}: incremento mínimo={delta.min():,.2f} € | "
        f"máximo={delta.max():,.2f} €"
    )

In [34]:
for nombre, objeto in globals().copy().items():
    if isinstance(objeto, pd.DataFrame):
        print(f"\n{nombre}: {objeto.shape}")
        print(objeto.columns.tolist())

In [35]:
# ============================================================
# COMPROBACIÓN 2/3 — INTEGRIDAD DEL TRACKING POR ACTIVIDAD
# ============================================================

# 1. Cobertura de actividades terminales
ids_terminal = set(
    df_master_operativo.loc[
        df_master_operativo["Is_Terminal"], "ID"
    ]
)

ids_tracking = set(df_activity_tracking["Activity_ID"].unique())

print("=== COBERTURA DE ACTIVIDADES ===")
print("Actividades terminales:", len(ids_terminal))
print("Actividades en tracking:", len(ids_tracking))
print("Terminales sin tracking:", len(ids_terminal - ids_tracking))
print("IDs de tracking fuera del maestro:", len(ids_tracking - set(df_master_operativo["ID"])))

# 2. Duplicados de clave actividad-periodo
duplicados = df_activity_tracking.duplicated(
    subset=["Activity_ID", "TP"]
).sum()

print("\n=== CLAVES ===")
print("Registros:", len(df_activity_tracking))
print("Claves duplicadas:", duplicados)

# 3. Cobertura esperada: 135 actividades x 120 periodos
esperados = len(ids_terminal) * df_tracking["TP"].nunique()

print("\n=== COBERTURA TEMPORAL ===")
print("Registros esperados:", esperados)
print("Registros reales:", len(df_act_terminal))
print("Diferencia:", len(df_act_terminal) - esperados)

# 4. Valores nulos en variables operativas
columnas = [
    "Actual_Cost", "EV", "PV", "Pct_Complete"
]

print("\n=== VALORES NULOS EN TRACKING TERMINAL ===")
print(df_act_terminal[columnas].isna().sum())

# 5. Comprobación de rangos
print("\n=== RANGOS ===")
print("Pct_Complete mínimo:", df_act_terminal["Pct_Complete"].min())
print("Pct_Complete máximo:", df_act_terminal["Pct_Complete"].max())
print("Costes negativos:", (df_act_terminal["Actual_Cost"] < 0).sum())
print("EV negativos:", (df_act_terminal["EV"] < 0).sum())
print("PV negativos:", (df_act_terminal["PV"] < 0).sum())

In [36]:

# ============================================================
# 06 — PREPARACIÓN DE PUNTOS DE DECISIÓN
# Base operativa para CPM y simulador
# ============================================================

#  Último periodo disponible
last_tp = df_activity_tracking["TP"].max()

df_latest = (
    df_act_terminal.loc[
        df_act_terminal["TP"].eq(last_tp)
    ]
    .sort_values("Activity_ID")
    .copy()
)

#  Variables de desempeño por actividad
df_latest["Schedule_Variance"] = (
    df_latest["EV"] - df_latest["PV"]
)

df_latest["Cost_Variance"] = (
    df_latest["EV"] - df_latest["Actual_Cost"]
)

df_latest["CPI_Activity"] = np.where(
    df_latest["Actual_Cost"] > 0,
    df_latest["EV"] / df_latest["Actual_Cost"],
    np.nan
)

#  Incorporar datos del maestro y materiales
master_cols = [
    "ID", "Name", "WBS",
    "Baseline_Start", "Baseline_End",
    "Duration_Plan_Days", "Total_Cost_Plan",
    "Is_Critical_Material", "Material_Type",
    "Material_Readiness_Date", "Supply_Delay_Days",
    "Material_Status", "Supply_Risk_Score",
    "Material_Alarm"
]

df_decision_base = df_latest.merge(
    df_master_operativo[master_cols],
    left_on="Activity_ID",
    right_on="ID",
    how="left",
    validate="one_to_one"
)

In [37]:
# ============================================================
# 06 — SELECCIÓN DE CANDIDATOS PARA PUNTOS DE DECISIÓN
# ============================================================
# Objetivo:
#   - Detectar incidencias en periodos históricos relevantes.
#   - No depender exclusivamente del último periodo (TP120).
#   - Seleccionar actividades con desviaciones de plazo, coste
#     o alertas de suministro.
#
# Requiere:
#   df_activity_tracking
#   df_master_operativo
#
# No modifica los datos originales.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURACIÓN DE PERIODOS HISTÓRICOS
# ------------------------------------------------------------

DECISION_PERIODS = [66, 80]

# Número máximo de candidatos por señal y periodo.
TOP_N = 10

# Tolerancia para evitar pequeñas diferencias numéricas.
TOL = 0.01


# ------------------------------------------------------------
# 2. PREPARAR ACTIVIDADES TERMINALES
# ------------------------------------------------------------

df_terminal_master = df_master_operativo.loc[
    df_master_operativo["Is_Terminal"].astype(bool)
].copy()

df_terminal_master["ID"] = pd.to_numeric(
    df_terminal_master["ID"], errors="coerce"
)

df_terminal_master = df_terminal_master.dropna(subset=["ID"])
df_terminal_master["ID"] = df_terminal_master["ID"].astype(int)

# Evitar duplicados en el maestro.
df_terminal_master = df_terminal_master.drop_duplicates(
    subset=["ID"]
)

# Columnas del maestro que realmente existen.
master_cols = [
    "ID",
    "Name",
    "WBS",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Is_Critical_Material",
    "Material_Type",
    "Material_Readiness_Date",
    "Supply_Delay_Days",
    "Material_Status",
    "Supply_Risk_Score",
    "Material_Alarm",
]

master_cols = [
    c for c in master_cols
    if c in df_terminal_master.columns
]

df_terminal_master = df_terminal_master[master_cols].copy()


# ------------------------------------------------------------
# 3. PREPARAR TRACKING HISTÓRICO
# ------------------------------------------------------------

df_hist = df_activity_tracking.copy()

df_hist["Activity_ID"] = pd.to_numeric(
    df_hist["Activity_ID"], errors="coerce"
)

df_hist["TP"] = pd.to_numeric(
    df_hist["TP"], errors="coerce"
)

df_hist = df_hist.dropna(
    subset=["Activity_ID", "TP"]
)

df_hist["Activity_ID"] = df_hist["Activity_ID"].astype(int)
df_hist["TP"] = df_hist["TP"].astype(int)

# Seleccionar únicamente los periodos históricos definidos.
df_hist = df_hist.loc[
    df_hist["TP"].isin(DECISION_PERIODS)
].copy()

# Solo actividades terminales: no sumar simultáneamente
# padres y descendientes.
terminal_ids = set(df_terminal_master["ID"])

df_hist = df_hist.loc[
    df_hist["Activity_ID"].isin(terminal_ids)
].copy()

# Renombrar columnas de tracking para evitar conflictos.
rename_tracking = {
    "Pct_Complete": "Pct_Complete",
    "EV": "EV",
    "PV": "PV",
    "Actual_Cost": "Actual_Cost",
}

required_tracking = [
    "Activity_ID",
    "TP",
    "Pct_Complete",
    "EV",
    "PV",
    "Actual_Cost",
]

missing_tracking = [
    c for c in required_tracking
    if c not in df_hist.columns
]

if missing_tracking:
    raise KeyError(
        "Faltan columnas en df_activity_tracking: "
        f"{missing_tracking}"
    )

for col in ["Pct_Complete", "EV", "PV", "Actual_Cost"]:
    df_hist[col] = pd.to_numeric(
        df_hist[col], errors="coerce"
    )


# ------------------------------------------------------------
# 4. UNIR TRACKING Y MAESTRO
# ------------------------------------------------------------

df_decision_base = df_hist.merge(
    df_terminal_master,
    left_on="Activity_ID",
    right_on="ID",
    how="left",
    validate="many_to_one",
)

# Variaciones calculadas con el tracking del periodo.
df_decision_base["Schedule_Variance"] = (
    df_decision_base["EV"] - df_decision_base["PV"]
)

df_decision_base["Cost_Variance"] = (
    df_decision_base["EV"]
    - df_decision_base["Actual_Cost"]
)

df_decision_base["CPI_Activity"] = np.where(
    df_decision_base["Actual_Cost"] > 0,
    df_decision_base["EV"]
    / df_decision_base["Actual_Cost"],
    np.nan,
)

# Actividad abierta en el periodo histórico.
df_decision_base["Is_Completed"] = (
    df_decision_base["Pct_Complete"] >= 0.999
)

df_decision_base["Is_Open"] = (
    ~df_decision_base["Is_Completed"]
)


# ------------------------------------------------------------
# 5. DEFINIR SEÑALES
# ------------------------------------------------------------

df_decision_base["Signal_Schedule"] = (
    df_decision_base["Schedule_Variance"] < -TOL
)

df_decision_base["Signal_Cost"] = (
    df_decision_base["Cost_Variance"] < -TOL
)

alarm = (
    df_decision_base["Material_Alarm"]
    .astype("string")
    .str.strip()
    .str.upper()
    if "Material_Alarm" in df_decision_base.columns
    else pd.Series(
        pd.NA,
        index=df_decision_base.index,
        dtype="string",
    )
)

df_decision_base["Signal_Material"] = (
    alarm.notna()
    & ~alarm.isin(["", "OK", "NONE", "NAN", "NO ALARM"])
)

# Las decisiones correctivas se orientan a actividades abiertas.
df_open = df_decision_base.loc[
    df_decision_base["Is_Open"]
].copy()

"""
# ------------------------------------------------------------
# 6. SELECCIONAR CANDIDATOS POR PERIODO Y SEÑAL
# ------------------------------------------------------------

candidate_frames = []

for tp in DECISION_PERIODS:

    df_tp = df_open.loc[
        df_open["TP"] == tp
    ].copy()

    # Desviaciones de plazo más negativas.
    schedule_candidates = (
        df_tp.loc[df_tp["Signal_Schedule"]]
        .sort_values("Schedule_Variance", ascending=True)
        .head(TOP_N)
        .copy()
    )

    if not schedule_candidates.empty:
        schedule_candidates["Selection_Reason"] = (
            "Desviación de plazo"
        )
        candidate_frames.append(schedule_candidates)

    # Desviaciones de coste más negativas.
    cost_candidates = (
        df_tp.loc[df_tp["Signal_Cost"]]
        .sort_values("Cost_Variance", ascending=True)
        .head(TOP_N)
        .copy()
    )

    if not cost_candidates.empty:
        cost_candidates["Selection_Reason"] = (
            "Desviación de coste"
        )
        candidate_frames.append(cost_candidates)

    # Alertas de materiales.
    material_candidates = (
        df_tp.loc[df_tp["Signal_Material"]]
        .copy()
    )

    if not material_candidates.empty:

        sort_cols = [
            c for c in [
                "Supply_Risk_Score",
                "Supply_Delay_Days",
            ]
            if c in material_candidates.columns
        ]

        if sort_cols:
            material_candidates = material_candidates.sort_values(
                sort_cols,
                ascending=[False] * len(sort_cols),
                na_position="last",
            )

        material_candidates["Selection_Reason"] = (
            "Alarma de materiales"
        )

        candidate_frames.append(material_candidates)

# ------------------------------------------------------------
# 7. CONSOLIDAR CANDIDATOS
# ------------------------------------------------------------

if candidate_frames:
    df_decision_candidates = pd.concat(
        candidate_frames,
        ignore_index=True,
    )

    # Una actividad puede aparecer por varias señales.
    # Consolidamos por periodo y actividad, conservando
    # todos los motivos de selección.
    group_cols = ["TP", "Activity_ID"]

    aggregation = {
        "Selection_Reason": lambda s: " | ".join(
            sorted(set(s.dropna().astype(str)))
        )
    }

    for col in [
        "Name",
        "WBS",
        "Pct_Complete",
        "EV",
        "PV",
        "Actual_Cost",
        "Schedule_Variance",
        "Cost_Variance",
        "CPI_Activity",
        "Material_Status",
        "Supply_Delay_Days",
        "Supply_Risk_Score",
        "Material_Alarm",
    ]:
        if col in df_decision_candidates.columns:
            aggregation[col] = "first"

    df_decision_candidates = (
        df_decision_candidates
        .groupby(group_cols, as_index=False)
        .agg(aggregation)
        .sort_values(
            ["TP", "Activity_ID"]
        )
        .reset_index(drop=True)
    )

else:
    df_decision_candidates = pd.DataFrame()


# ------------------------------------------------------------
# 8. MOSTRAR RESULTADOS
# ------------------------------------------------------------

print("=" * 80)
print("CANDIDATOS HISTÓRICOS PARA PUNTOS DE DECISIÓN")
print("=" * 80)

print(f"Periodos analizados: {DECISION_PERIODS}")
print(
    "Actividades terminales en el maestro: "
    f"{len(df_terminal_master)}"
)
print(
    "Registros históricos de actividades abiertas: "
    f"{len(df_open)}"
)
print(
    "Candidatos únicos (periodo + actividad): "
    f"{len(df_decision_candidates)}"
)

if not df_decision_candidates.empty:

    display_cols = [
        "TP",
        "Activity_ID",
        "Name",
        "WBS",
        "Pct_Complete",
        "EV",
        "PV",
        "Actual_Cost",
        "Schedule_Variance",
        "Cost_Variance",
        "CPI_Activity",
        "Material_Status",
        "Supply_Delay_Days",
        "Supply_Risk_Score",
        "Material_Alarm",
        "Selection_Reason",
    ]

    display_cols = [
        c for c in display_cols
        if c in df_decision_candidates.columns
    ]

    display(
        df_decision_candidates[display_cols]
    )

else:
    print(
        "\nNo se han identificado candidatos abiertos "
        "con las señales configuradas en estos periodos."
    )


# ------------------------------------------------------------
# 9. RESUMEN DE SEÑALES EN LOS PERIODOS ANALIZADOS
# ------------------------------------------------------------

print("\n=== RESUMEN DE SEÑALES ===")

if not df_open.empty:

    summary_signals = (
        df_open.groupby("TP")
        .agg(
            Actividades_Abiertas=("Activity_ID", "nunique"),
            Desviacion_Plazo=("Signal_Schedule", "sum"),
            Desviacion_Coste=("Signal_Cost", "sum"),
            Alarma_Materiales=("Signal_Material", "sum"),
        )
        .reindex(DECISION_PERIODS, fill_value=0)
    )

    display(summary_signals)

else:
    print(
        "No hay actividades abiertas en los periodos seleccionados."
    )

"""

# ------------------------------------------------------------
# 6. SELECCIONAR CANDIDATOS POR PERIODO Y SEÑAL
# ------------------------------------------------------------

candidate_frames = []

for tp in DECISION_PERIODS:

    df_tp = df_open.loc[df_open["TP"] == tp].copy()

    # Recalcular señales en el subconjunto para evitar
    # arrastrar motivos de selección incorrectos.
    df_tp["Signal_Schedule"] = (
        df_tp["Schedule_Variance"] < -TOL
    )

    df_tp["Signal_Cost"] = (
        df_tp["Cost_Variance"] < -TOL
    )

    # A. Desviaciones reales de plazo
    schedule_candidates = (
        df_tp.loc[
            df_tp["Signal_Schedule"]
            & df_tp["Schedule_Variance"].notna()
        ]
        .sort_values("Schedule_Variance", ascending=True)
        .head(TOP_N)
        .copy()
    )

    if not schedule_candidates.empty:
        schedule_candidates["Selection_Reason"] = (
            "Desviación de plazo"
        )
        candidate_frames.append(schedule_candidates)

    # B. Desviaciones reales de coste
    cost_candidates = (
        df_tp.loc[
            df_tp["Signal_Cost"]
            & df_tp["Cost_Variance"].notna()
        ]
        .sort_values("Cost_Variance", ascending=True)
        .head(TOP_N)
        .copy()
    )

    if not cost_candidates.empty:
        cost_candidates["Selection_Reason"] = (
            "Desviación de coste"
        )
        candidate_frames.append(cost_candidates)

    # C. Alertas de materiales
    material_candidates = df_tp.loc[
        df_tp["Signal_Material"]
    ].copy()

    if not material_candidates.empty:

        sort_cols = [
            c for c in [
                "Supply_Risk_Score",
                "Supply_Delay_Days",
            ]
            if c in material_candidates.columns
        ]

        if sort_cols:
            material_candidates = material_candidates.sort_values(
                sort_cols,
                ascending=[False] * len(sort_cols),
                na_position="last",
            )

        material_candidates["Selection_Reason"] = (
            "Alarma de materiales"
        )

        candidate_frames.append(material_candidates)


# ------------------------------------------------------------
# 7. CONSOLIDAR SIN PERDER LOS MOTIVOS DE SELECCIÓN
# ------------------------------------------------------------

if candidate_frames:

    df_candidates_raw = pd.concat(
        candidate_frames,
        ignore_index=True,
    )

    # Agrupar por actividad y periodo, combinando motivos.
    group_cols = ["TP", "Activity_ID"]

    aggregation = {
        "Selection_Reason": lambda s: " | ".join(
            sorted(set(s.dropna().astype(str)))
        )
    }

    for col in [
        "Name",
        "WBS",
        "Pct_Complete",
        "EV",
        "PV",
        "Actual_Cost",
        "Schedule_Variance",
        "Cost_Variance",
        "CPI_Activity",
        "Material_Status",
        "Supply_Delay_Days",
        "Supply_Risk_Score",
        "Material_Alarm",
    ]:
        if col in df_candidates_raw.columns:
            aggregation[col] = "first"

    df_decision_candidates = (
        df_candidates_raw
        .groupby(group_cols, as_index=False)
        .agg(aggregation)
        .sort_values(["TP", "Activity_ID"])
        .reset_index(drop=True)
    )

else:
    df_decision_candidates = pd.DataFrame()


# ------------------------------------------------------------
# 8. COMPROBACIONES DEL RESULTADO
# ------------------------------------------------------------

if not df_decision_candidates.empty:

    # Un candidato seleccionado por plazo debe tener
    # una desviación de plazo realmente negativa.
    mask_schedule = (
        df_decision_candidates["Selection_Reason"]
        .str.contains("Desviación de plazo", regex=False)
    )

    assert (
        df_decision_candidates.loc[
            mask_schedule, "Schedule_Variance"
        ] < -TOL
    ).all(), "Hay candidatos de plazo sin desviación negativa."

    # Un candidato seleccionado por coste debe tener
    # una desviación de coste realmente negativa.
    mask_cost = (
        df_decision_candidates["Selection_Reason"]
        .str.contains("Desviación de coste", regex=False)
    )

    assert (
        df_decision_candidates.loc[
            mask_cost, "Cost_Variance"
        ] < -TOL
    ).all(), "Hay candidatos de coste sin desviación negativa."

print("Candidatos históricos:", len(df_decision_candidates))
print(
    df_decision_candidates[
        [
            "TP",
            "Activity_ID",
            "Name",
            "Schedule_Variance",
            "Cost_Variance",
            "Material_Alarm",
            "Selection_Reason",
        ]
    ].head(20).to_string(index=False)
)

In [38]:
"""
# ============================================================
# 06 — SELECCIÓN OPERATIVA DE CANDIDATOS
# ============================================================

# 1. Estado de ejecución
df_decision_base["Pct_Complete"] = pd.to_numeric(
    df_decision_base["Pct_Complete"],
    errors="coerce"
)

df_decision_base["Is_Completed"] = (
    df_decision_base["Pct_Complete"] >= 0.999
)

df_decision_base["Is_In_Progress"] = (
    df_decision_base["Pct_Complete"].between(
        0, 0.999, inclusive="left"
    )
)

# 2. Señales de desempeño
df_decision_base["Signal_Schedule"] = (
    df_decision_base["Schedule_Variance"] < -0.01
)

df_decision_base["Signal_Cost"] = (
    df_decision_base["Cost_Variance"] < -0.01
)

alarm = (
    df_decision_base["Material_Alarm"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df_decision_base["Signal_Material"] = (
    alarm.notna()
    & ~alarm.isin(["", "ok", "none", "nan", "no alarm"])
)

# 3. Candidatos a intervención futura:
# solo actividades no completadas.
df_open = df_decision_base.loc[
    ~df_decision_base["Is_Completed"]
].copy()

# Ranking por desempeño, sin umbrales arbitrarios.
ids_cost = set(
    df_open.loc[
        df_open["Signal_Cost"]
    ]
    .sort_values("Cost_Variance")
    .head(10)["Activity_ID"]
)

ids_schedule = set(
    df_open.loc[
        df_open["Signal_Schedule"]
    ]
    .sort_values("Schedule_Variance")
    .head(10)["Activity_ID"]
)

ids_material = set(
    df_open.loc[
        df_open["Signal_Material"]
    ]["Activity_ID"]
)

candidate_ids = ids_cost | ids_schedule | ids_material

df_decision_candidates = df_open.loc[
    df_open["Activity_ID"].isin(candidate_ids)
].copy()

# 4. Explicar por qué entra cada actividad.
def decision_reasons(row):
    reasons = []

    if row["Signal_Schedule"]:
        reasons.append("Schedule variance")

    if row["Signal_Cost"]:
        reasons.append("Cost variance")

    if row["Signal_Material"]:
        reasons.append("Material alarm")

    return " | ".join(reasons)

df_decision_candidates["Selection_Reason"] = (
    df_decision_candidates.apply(
        decision_reasons,
        axis=1
    )
)

# 5. Orden de lectura: alertas materiales y peores desviaciones.
df_decision_candidates = (
    df_decision_candidates
    .sort_values(
        [
            "Signal_Material",
            "Signal_Schedule",
            "Signal_Cost",
            "Cost_Variance"
        ],
        ascending=[False, False, False, True]
    )
    .reset_index(drop=True)
)

# 6. Mostrar resultado
cols_view = [
    "Activity_ID",
    "Name",
    "WBS",
    "Pct_Complete",
    "EV",
    "PV",
    "Actual_Cost",
    "Schedule_Variance",
    "Cost_Variance",
    "CPI_Activity",
    "Material_Status",
    "Supply_Delay_Days",
    "Supply_Risk_Score",
    "Material_Alarm",
    "Selection_Reason"
]

print("=== CANDIDATOS ABIERTOS ===")
print("Actividades terminales:", len(df_decision_base))
print("Actividades completadas:", df_decision_base["Is_Completed"].sum())
print("Actividades no completadas:", len(df_open))
print("Candidatos únicos:", len(df_decision_candidates))

display(df_decision_candidates[cols_view])

print("\n=== RESUMEN DE SEÑALES EN ACTIVIDADES ABIERTAS ===")
print(
    "Desviación de plazo:",
    int(df_open["Signal_Schedule"].sum())
)
print(
    "Desviación de coste:",
    int(df_open["Signal_Cost"].sum())
)
print(
    "Alarma de materiales:",
    int(df_open["Signal_Material"].sum())
)
"""

In [39]:
# Actividades seleccionadas por más de un motivo
df_decision_candidates.loc[
    df_decision_candidates["Selection_Reason"].str.contains(r"\|", regex=True),
    ["TP", "Activity_ID", "Name", "Selection_Reason"]
].to_string(index=False)

In [40]:

# ============================================================
# 06 — EXPORTACIÓN DE RESULTADOS
# ============================================================

DIR_06.mkdir(parents=True, exist_ok=True)

# Libro de trabajo para los siguientes notebooks
output_excel = DIR_06 / "06_Decision_Points_Base.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:

    df_decision_base.to_excel(
        writer,
        sheet_name="Activity_Performance",
        index=False
    )

    df_decision_candidates.to_excel(
        writer,
        sheet_name="Decision_Candidates",
        index=False
    )

    df_tracking.to_excel(
        writer,
        sheet_name="Project_EVM",
        index=False
    )

    df_check.to_excel(
        writer,
        sheet_name="EVM_Reconciliation",
        index=False
    )

    df_master_operativo.to_excel(
        writer,
        sheet_name="Activity_Master",
        index=False
    )

# CSV específicos para facilitar la integración posterior
df_decision_candidates.to_csv(
    DIR_06 / "Decision_Candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

df_master_operativo.to_csv(
    DIR_06 / "Activity_Master_CPM_Input.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=== EXPORTACIÓN COMPLETADA ===")
print("Excel:", output_excel)
print("Candidatos:", DIR_06 / "Decision_Candidates.csv")
print("Maestro para CPM:", DIR_06 / "Activity_Master_CPM_Input.csv")
